# Masking Inspector — SmolLM2-135M-Instruct

Compare **legacy** (`instruction_template` + `response_template`) vs **span-based** (`assistant_turn` with `end_of_turn_template`) masking using the SmolLM2 tokenizer.

SmolLM2 uses ChatML format but keeps `role="tool"` as `<|im_start|>tool` — the legacy collator will **not** match this as an instruction boundary, so tool results get incorrectly trained on (same problem as Llama).

In [ ]:
import json
import warnings

from transformers import AutoTokenizer

from oumi.core.collators.text_completions_collator_with_padding import (
    TextCompletionsCollatorWithPadding,
)

MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
print(f"Loaded tokenizer: {MODEL}")
print(
    f"Vocab size: {tokenizer.vocab_size}, pad={tokenizer.pad_token_id}, eos={tokenizer.eos_token_id}"
)

In [ ]:
# Show how SmolLM2 formats each role
demo = [
    {"role": "system", "content": "You are helpful."},
    {"role": "user", "content": "Call a tool"},
    {"role": "assistant", "content": '{"name": "do_thing"}'},
    {"role": "tool", "content": "result=42"},
    {"role": "assistant", "content": "The answer is 42."},
]
print(tokenizer.apply_chat_template(demo, tokenize=False, add_generation_prompt=False))
print()
print("SmolLM2 keeps tool as <|im_start|>tool — NOT converted to user!")
print("Legacy collator will NOT match this → trains on tool results incorrectly.")

In [ ]:
def make_old_collator(tokenizer):
    """Legacy collator: instruction_template + response_template."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", DeprecationWarning)
        return TextCompletionsCollatorWithPadding(
            tokenizer=tokenizer,
            instruction_template="<|im_start|>user\n",
            response_template="<|im_start|>assistant\n",
        )


def make_new_collator(tokenizer):
    """Span-based collator: assistant_turn with end_of_turn."""
    return TextCompletionsCollatorWithPadding(
        tokenizer=tokenizer,
        response_template="<|im_start|>assistant\n",
        end_of_turn_template="<|im_end|>",
        masking_method="assistant_turn",
    )


old_collator = make_old_collator(tokenizer)
new_collator = make_new_collator(tokenizer)
print("Old collator masking_method:", old_collator._default_collator.masking_method)
print("New collator masking_method:", new_collator._default_collator.masking_method)

In [ ]:
from IPython.display import HTML, display


def visualize_masking(input_ids, labels, tokenizer, title=""):
    """Render tokens color-coded: green=unmasked (trained on), gray=masked."""
    html = f"<h4>{title}</h4><div style='font-family:monospace; line-height:1.8; white-space:pre-wrap;'>"
    for tid, lab in zip(input_ids, labels):
        tok_str = (
            tokenizer.decode([tid])
            .replace("<", "&lt;")
            .replace(">", "&gt;")
            .replace(" ", "\u00b7")
        )
        if lab == -100:
            html += f"<span style='color:#999; background:#f0f0f0;'>{tok_str}</span>"
        else:
            html += f"<span style='color:#000; background:#b5f5b5; font-weight:bold;'>{tok_str}</span>"
    html += "</div>"
    display(HTML(html))


def compare_masking(messages, tokenizer, old_collator, new_collator, title=""):
    """Tokenize messages, run both collators, show side-by-side."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    input_ids = tokenizer.encode(text, add_special_tokens=False)
    batch = [{"input_ids": input_ids}]

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        old_out = old_collator(batch)
        new_out = new_collator(batch)

    old_labels = old_out["labels"][0].tolist()
    new_labels = new_out["labels"][0].tolist()
    ids = old_out["input_ids"][0].tolist()

    old_unmasked = sum(1 for x in old_labels if x != -100)
    new_unmasked = sum(1 for x in new_labels if x != -100)
    diffs = sum(1 for a, b in zip(old_labels, new_labels) if (a == -100) != (b == -100))

    print(f"=== {title} ===")
    print(f"Total tokens: {len(ids)}")
    print(
        f"Old unmasked: {old_unmasked}, New unmasked: {new_unmasked}, Diff positions: {diffs}"
    )

    visualize_masking(ids, old_labels, tokenizer, title=f"{title} — OLD (legacy)")
    visualize_masking(ids, new_labels, tokenizer, title=f"{title} — NEW (span-based)")

    if diffs > 0:
        print(f"\nDiffering tokens ({diffs}):")
        for i, (a, b) in enumerate(zip(old_labels, new_labels)):
            if (a == -100) != (b == -100):
                tok = tokenizer.decode([ids[i]])
                old_s = "TRAINED" if a != -100 else "masked"
                new_s = "TRAINED" if b != -100 else "masked"
                print(f"  pos {i}: {repr(tok):30s}  old={old_s:8s}  new={new_s}")
    return old_labels, new_labels

## 1. Toy Data

In [ ]:
# Single-turn
toy_single = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2+2?"},
    {"role": "assistant", "content": "The answer is 4."},
]
compare_masking(toy_single, tokenizer, old_collator, new_collator, "Toy: single-turn");

In [ ]:
# Multi-turn
toy_multi = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2+2?"},
    {"role": "assistant", "content": "The answer is 4."},
    {"role": "user", "content": "And 3+3?"},
    {"role": "assistant", "content": "That would be 6."},
]
compare_masking(toy_multi, tokenizer, old_collator, new_collator, "Toy: multi-turn");

In [ ]:
# Tool-calling — EXPECT BIG DIFF
# SmolLM2 keeps <|im_start|>tool, so legacy collator won't mask tool results
toy_tool = [
    {"role": "system", "content": "You are a helpful assistant with tool access."},
    {"role": "user", "content": "What's the weather in SF?"},
    {
        "role": "assistant",
        "content": '<tool_call>{"name": "get_weather", "arguments": {"city": "SF"}}</tool_call>',
    },
    {
        "role": "tool",
        "content": '<tool_response>{"temp": "65F", "condition": "sunny"}</tool_response>',
    },
    {"role": "assistant", "content": "It's 65F and sunny in San Francisco."},
]
compare_masking(
    toy_tool,
    tokenizer,
    old_collator,
    new_collator,
    "Toy: tool-calling (EXPECT BIG DIFF)",
);

In [ ]:
# Multiple sequential tool calls
toy_multi_tool = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Compare weather in SF and NYC"},
    {
        "role": "assistant",
        "content": '<tool_call>{"name": "get_weather", "arguments": {"city": "SF"}}</tool_call>',
    },
    {
        "role": "tool",
        "content": '<tool_response>{"temp": "65F", "condition": "sunny"}</tool_response>',
    },
    {
        "role": "assistant",
        "content": '<tool_call>{"name": "get_weather", "arguments": {"city": "NYC"}}</tool_call>',
    },
    {
        "role": "tool",
        "content": '<tool_response>{"temp": "45F", "condition": "cloudy"}</tool_response>',
    },
    {"role": "assistant", "content": "SF is 65F sunny, NYC is 45F cloudy."},
]
compare_masking(
    toy_multi_tool,
    tokenizer,
    old_collator,
    new_collator,
    "Toy: multi-tool (EXPECT BIGGER DIFF)",
);

## 2. TatQA Data

TatQA is single-turn (system/user/assistant, no tool calls), so diffs should be minimal.

In [ ]:
TATQA_PATH = "/data/shanghong/oumi-pr2/tool_call_project/data/train_final_llama3.3_70b_instruct_max2048.jsonl"
with open(TATQA_PATH) as f:
    tatqa_samples = [json.loads(line) for line in f.readlines()[:5]]
print(f"Loaded {len(tatqa_samples)} TatQA samples")

for i, sample in enumerate(tatqa_samples[:3]):
    msgs = sample["messages"]
    preview = msgs[-1]["content"][:80]
    print(f"\nSample {i}: {len(msgs)} messages, last assistant: {preview}...")
    compare_masking(msgs, tokenizer, old_collator, new_collator, f"TatQA sample {i}")

## 3. Hermes Data

Hermes has tool-calling conversations. SmolLM2 keeps `<|im_start|>tool`, so the legacy collator can't see tool results as instruction boundaries. **Expect large masking differences.**

In [ ]:
HERMES_PATH = "/data/shanghong/oumi-pr2/tool_call_project/data/hermes_reasoning_tool_use_train_split_train_filtered.jsonl"
with open(HERMES_PATH) as f:
    hermes_samples = [json.loads(line) for line in f.readlines()[:5]]
print(f"Loaded {len(hermes_samples)} Hermes samples")

for i, sample in enumerate(hermes_samples[:3]):
    msgs = sample["messages"]
    roles = [m["role"] for m in msgs]
    print(f"\nSample {i}: {len(msgs)} messages, roles: {roles}")
    compare_masking(msgs, tokenizer, old_collator, new_collator, f"Hermes sample {i}")

## Summary: Aggregate Diff Stats

In [ ]:
def diff_stats(data_path, n=50):
    with open(data_path) as f:
        samples = [json.loads(line) for line in f.readlines()[:n]]
    total_diffs = 0
    total_tokens = 0
    total_old_unmasked = 0
    total_new_unmasked = 0
    legacy_trains_on_extra = 0
    span_trains_on_extra = 0
    for sample in samples:
        msgs = sample["messages"]
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
        input_ids = tokenizer.encode(text, add_special_tokens=False)
        batch = [{"input_ids": input_ids}]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            old_out = old_collator(batch)
            new_out = new_collator(batch)
        old_labels = old_out["labels"][0].tolist()
        new_labels = new_out["labels"][0].tolist()
        total_tokens += len(input_ids)
        total_old_unmasked += sum(1 for x in old_labels if x != -100)
        total_new_unmasked += sum(1 for x in new_labels if x != -100)
        for a, b in zip(old_labels, new_labels):
            if (a == -100) != (b == -100):
                total_diffs += 1
                if a != -100:
                    legacy_trains_on_extra += 1
                else:
                    span_trains_on_extra += 1
    print(f"  Samples: {len(samples)}")
    print(f"  Total tokens: {total_tokens}")
    print(f"  Old unmasked: {total_old_unmasked}")
    print(f"  New unmasked: {total_new_unmasked}")
    print(
        f"  Differing positions: {total_diffs} ({total_diffs / total_tokens * 100:.2f}%)"
    )
    print(f"    Legacy trains on (span masks): {legacy_trains_on_extra}")
    print(f"    Span trains on (legacy masks): {span_trains_on_extra}")


print("TatQA (no tool calls — expect small diff):")
diff_stats(TATQA_PATH)
print("\nHermes (has tool calls — expect large diff):")
diff_stats(HERMES_PATH)